# Extracting the features 
## SMILES to MACCS + Morgan + Mordred descriptors

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import requests
import time

from rdkit import Chem
from rdkit.Chem import MACCSkeys, AllChem, rdFingerprintGenerator
from mordred import Calculator, descriptors

## 1. Config

In [2]:
INPUT_FILE    = r"c:\Users\hp\Desktop\stage etis\dataset\7Q27.normalized.signatures.csv"
OUTPUT_FILE   = r"c:\Users\hp\Desktop\stage etis\dataset\dataset_all_features.csv"

EXCLUDED_ITEMS  = {"Blanc_080426", "allumage_080426"}
NAN_THRESHOLD   = 0.20
CORR_THRESHOLD  = 0.95
MORGAN_RADIUS   = 2
MORGAN_NBITS    = 512

## 2. SMILES Fetching 


In [6]:
#hna anmappiw names dyal lmolecule li machi nfsoum fdictionnaire dyal names li kayn f api of pubchem

names_map = {
    "A-Pinene": "Alpha-Pinene",
    "S-limonene": "(-)-Limonene",
    "R-limonene": "(+)-Limonene"
}


def clean_name(name):

    base_name = name.split('_')[0].strip()
    base_name = base_name.replace(" ", "-")

    # Map to PubChem name if it exists in the dictionary
    pubchem_name = names_map.get(base_name, base_name)
    return pubchem_name


def get_smiles(name):
    """Fetches SMILES from PubChem."""
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}/property/IsomericSMILES/JSON"
    r = requests.get(url)
    if r.status_code == 200:
        return r.json()['PropertyTable']['Properties'][0]['SMILES']
    return None

In [7]:
df = pd.read_csv(INPUT_FILE, sep=';')
unique_items = df['item'].unique()

smiles_mapping = {}

for raw_item in unique_items:
    if 'blanc' in raw_item.lower() or 'allumage' in raw_item.lower():
        continue

    search_name = clean_name(raw_item)
    smiles = get_smiles(search_name)

    smiles_mapping[raw_item] = smiles

    if smiles:
        print(f"Success | Raw: {raw_item:<20} | Searched: {search_name:<15} | SMILES: {smiles}")
    else:
        print(f"Failed  | Raw: {raw_item:<20} | Searched: {search_name:<15}")

    time.sleep(0.2)

df['SMILES'] = df['item'].map(smiles_mapping)

print(f"\nSMILES fetched for {df['SMILES'].notna().sum()}/{len(df)} rows")
print(df[['item', 'SMILES']].drop_duplicates())

Success | Raw: Ocimene_080426       | Searched: Ocimene         | SMILES: CC(C)/C=C/C=C(\C)/C=C
Success | Raw: Delta 3 Carene_080426 | Searched: Delta-3-Carene  | SMILES: CC1=CCC2C(C1)C2(C)C
Success | Raw: Linalol_080426       | Searched: Linalol         | SMILES: CC(=CCCC(C)(C=C)O)C
Success | Raw: A Pinene_080426      | Searched: Alpha-Pinene    | SMILES: CC1=CCC2CC1C2(C)C
Success | Raw: S limonene_080426    | Searched: (-)-Limonene    | SMILES: CC1=CC[C@H](CC1)C(=C)C
Success | Raw: R limonene_080426    | Searched: (+)-Limonene    | SMILES: CC1=CC[C@@H](CC1)C(=C)C

SMILES fetched for 18/21 rows
                     item                   SMILES
0          Ocimene_080426    CC(C)/C=C/C=C(\C)/C=C
3   Delta 3 Carene_080426      CC1=CCC2C(C1)C2(C)C
6          Linalol_080426      CC(=CCCC(C)(C=C)O)C
9         A Pinene_080426        CC1=CCC2CC1C2(C)C
12      S limonene_080426   CC1=CC[C@H](CC1)C(=C)C
15      R limonene_080426  CC1=CC[C@@H](CC1)C(=C)C
18           Blanc_080426               

## 3. MACCS Fingerprints

In [8]:
def smiles_to_maccs(smiles):
    # in the documentation they work with objects of the rdchem.Mol class so nconvertiwha lobjects dyal dik lclass
    mol = Chem.MolFromSmiles(smiles)

    fp = MACCSkeys.GenMACCSKeys(mol)
    return list(fp)

In [9]:
df_clean = df[~df['item'].isin(EXCLUDED_ITEMS) & df['SMILES'].notna()].copy()
unique_smiles = df_clean[['item', 'SMILES']].drop_duplicates(subset='SMILES')

print("Computing MACCS fingerprints for each unique molecule:\n")
maccs_data = []

for s in unique_smiles['SMILES']:
    bits = smiles_to_maccs(s)
    if bits:
        entry = {'SMILES': s}
        entry.update({f'MACCS_{i:03d}': bits[i] for i in range(1, 167)})
        maccs_data.append(entry)

maccs_df = pd.DataFrame(maccs_data)
print(f"MACCS done: {len(maccs_df)} molecules × {len(maccs_df.columns)-1} bits")
print(maccs_df[['SMILES']].head())

Computing MACCS fingerprints for each unique molecule:

MACCS done: 6 molecules × 166 bits
                   SMILES
0   CC(C)/C=C/C=C(\C)/C=C
1     CC1=CCC2C(C1)C2(C)C
2     CC(=CCCC(C)(C=C)O)C
3       CC1=CCC2CC1C2(C)C
4  CC1=CC[C@H](CC1)C(=C)C


## 4. Morgan Fingerprints

In [10]:
unique_mols = df_clean[['item', 'SMILES']].drop_duplicates().dropna(subset=['SMILES'])

generator = rdFingerprintGenerator.GetMorganGenerator(radius=MORGAN_RADIUS, fpSize=MORGAN_NBITS)

fp_records = []
for _, row in unique_mols.iterrows():
    mol = Chem.MolFromSmiles(row['SMILES'])
    if mol:

        # Generate the bit vector as a list of integers
        bits = list(generator.GetFingerprintAsNumPy(mol))

        # Map each bit to a column name like morgan_0, morgan_1, ..., morgan_511
        feature_dict = {f'morgan_{i}': val for i, val in enumerate(bits)}
        feature_dict['SMILES'] = row['SMILES']
        fp_records.append(feature_dict)

morgan_df = pd.DataFrame(fp_records)
print(f"Morgan done: {len(morgan_df)} molecules × {MORGAN_NBITS} bits")
print(morgan_df[['SMILES']].head())

Morgan done: 6 molecules × 512 bits
                   SMILES
0   CC(C)/C=C/C=C(\C)/C=C
1     CC1=CCC2C(C1)C2(C)C
2     CC(=CCCC(C)(C=C)O)C
3       CC1=CCC2CC1C2(C)C
4  CC1=CC[C@H](CC1)C(=C)C


## 5. Mordred Descriptors

In [11]:
def smiles_to_3d_mol(smiles):

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = Chem.AddHs(mol)
    result = AllChem.EmbedMolecule(mol, randomSeed=42)
    if result == -1:
        return None
    AllChem.MMFFOptimizeMolecule(mol)
    mol = Chem.RemoveHs(mol)
    return mol

In [12]:
calc = Calculator(descriptors, ignore_3D=False)

mordred_rows = []
for _, row in unique_smiles.iterrows():
    mol = smiles_to_3d_mol(row['SMILES'])
    if mol is None:
        print(f"  WARNING: 3D embedding failed for {row['item']} → {row['SMILES']}")
        mordred_rows.append({'SMILES': row['SMILES']})
        continue
    result = calc(mol)
    desc_dict = {str(k): v for k, v in result.items()}
    desc_dict['SMILES'] = row['SMILES']
    mordred_rows.append(desc_dict)

mordred_df = pd.DataFrame(mordred_rows)

desc_cols = [c for c in mordred_df.columns if c != 'SMILES']
mordred_df[desc_cols] = mordred_df[desc_cols].apply(pd.to_numeric, errors='coerce')

print(f"Raw Mordred descriptors: {len(desc_cols)}")

Raw Mordred descriptors: 1826


### Filtering

#### Dropping the descriptors with too many NaN values

In [13]:
nan_frac = mordred_df[desc_cols].isna().mean()
keep = nan_frac[nan_frac <= NAN_THRESHOLD].index.tolist()
dropped_nan = len(desc_cols) - len(keep)
print(f"Dropped (>{NAN_THRESHOLD*100:.0f}% NaN):       {dropped_nan}")
desc_cols = keep

# for the ones left with little Nan values i replaced the Nan with the median
mordred_df[desc_cols] = mordred_df[desc_cols].fillna(mordred_df[desc_cols].median())

Dropped (>20% NaN):       386


#### Dropping the descriptors with 0 variance

In [14]:
variance = mordred_df[desc_cols].var()
keep = variance[variance > 0].index.tolist()
dropped_var = len(desc_cols) - len(keep)
print(f"Dropped (zero variance):      {dropped_var}")
desc_cols = keep

Dropped (zero variance):      352


#### Drop the highly correlated descriptor pairs

In [15]:
corr_matrix = mordred_df[desc_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > CORR_THRESHOLD)]
desc_cols = [c for c in desc_cols if c not in to_drop]
print(f"Dropped (corr > {CORR_THRESHOLD}):     {len(to_drop)}")

print(f"Final Mordred descriptor count:       {len(desc_cols)}")

Dropped (corr > 0.95):     1040
Final Mordred descriptor count:       48


In [16]:
mordred_clean = mordred_df[['SMILES'] + desc_cols]

## 6. Merge All Features & Save

In [17]:
result_df = df.merge(maccs_df,     on='SMILES', how='left')
result_df = result_df.merge(morgan_df,    on='SMILES', how='left')
result_df = result_df.merge(mordred_clean, on='SMILES', how='left')

result_df.to_csv(OUTPUT_FILE, index=False, sep=';')

print(f"\n=== Final dataset ===")
print(f"Rows:    {len(result_df)}")
print(f"Columns: {len(result_df.columns)}")
print(f"  → Base columns:       {len(df.columns)}")
print(f"  → MACCS bits:         166")
print(f"  → Morgan bits:        {MORGAN_NBITS}")
print(f"  → Mordred descriptors:{len(desc_cols)}")
print(f"\nSaved to: {OUTPUT_FILE}")
print(result_df.head())


=== Final dataset ===
Rows:    21
Columns: 750
  → Base columns:       24
  → MACCS bits:         166
  → Morgan bits:        512
  → Mordred descriptors:48

Saved to: c:\Users\hp\Desktop\stage etis\dataset\dataset_all_features.csv
   record_seq_num  unix_timestamp_sec      run_id    run_name  \
0               0        1.775659e+09  2604081640  2604081640   
1               1        1.775658e+09  2604081618  2604081618   
2               2        1.775657e+09  2604081556  2604081556   
3               3        1.775654e+09  2604081518  2604081518   
4               4        1.775653e+09  2604081459  2604081459   

                device_id                   item  cycle  baseline_start_sec  \
0  NOA222-10002-3H5QU_fsp         Ocimene_080426      1                   0   
1  NOA222-10002-3H5QU_fsp         Ocimene_080426      1                   0   
2  NOA222-10002-3H5QU_fsp         Ocimene_080426      1                   0   
3  NOA222-10002-3H5QU_fsp  Delta 3 Carene_080426      1     